In [1]:
# drive allocation
import os
from google.colab import drive
from datasets import load_dataset, Dataset, load_from_disk
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
BASE_PATH = r"CSCI 5980 8980 Project"

ORIGIN_PATH = f"/content/drive/MyDrive/{BASE_PATH}/Notebooks/MICE_Output/"
OUTPUT_PATH = f"/content/drive/MyDrive/{BASE_PATH}/Notebooks/MICE_Output/eval-2.5k/"

layers_full_path = os.path.join(OUTPUT_PATH, "layers_full.csv")

summary_full_path = os.path.join(OUTPUT_PATH, "summary_full.csv")

In [3]:
summaryA = os.path.join(ORIGIN_PATH, "lite_results_summary_judged_trek.csv")
summaryB  = os.path.join(ORIGIN_PATH, "results_summary_judged_brandon.csv")
summaryC = os.path.join(ORIGIN_PATH, "results_summary_judged_justin.csv")

layersA = os.path.join(ORIGIN_PATH, "lite_results_layers_trek.csv")
layersB = os.path.join(ORIGIN_PATH, "results_layers_brandon.csv")
layersC = os.path.join(ORIGIN_PATH, "results_layers_justin.csv")

In [6]:
summary_dfA = pd.read_csv(summaryA)
summary_dfB = pd.read_csv(summaryB)
summary_dfC = pd.read_csv(summaryC)

layers_dfA = pd.read_csv(layersA)
layers_dfB = pd.read_csv(layersB)
layers_dfC = pd.read_csv(layersC)

# combine dfA and dfB and dfC
summary_df = pd.concat([summary_dfA, summary_dfB, summary_dfC])
layers_df = pd.concat([layers_dfA, layers_dfB, layers_dfC])

summary_df.to_csv(summary_full_path, index=False)
layers_df.to_csv(layers_full_path, index=False)

In [7]:
# Pivot & Merge
layer_pivot = (
    layers_df.pivot_table(index="question_id", columns="layer", values="f1")
    .sort_index(axis=1)
    .reset_index()
)
layer_pivot.columns = ["question_id"] + [f"f1_layer_{c}" for c in layer_pivot.columns[1:]]

df = summary_df.merge(layer_pivot, on="question_id")

# Define Features and Stratification
layer_cols = [c for c in df.columns if c.startswith("f1_layer_")]
feature_cols = layer_cols + ["normalized_log_confidence"]

# Create a combined key for balanced 'type' AND 'judge_decision'
df['stratify_key'] = df['type'].astype(str) + "_" + df['judge_decision'].astype(str)

# Stratified Splitting
# Split 1: 80% Train+Val, 20% Test
df_tv, df_test = train_test_split(
    df, test_size=0.20, random_state=42, stratify=df['stratify_key']
)

# Split 2: Of that 80%, 25% for Val (results in 60% Train, 20% Val total)
df_train, df_val = train_test_split(
    df_tv, test_size=0.25, random_state=42, stratify=df_tv['stratify_key']
)

# Extract Arrays
def get_xyt(target_df):
    X = target_df[feature_cols].values.astype(np.float32)
    y = target_df["judge_decision"].values.astype(int)
    act_type = target_df["type"].values.astype(str)
    pred_type = target_df["predicted_type"].values.astype(str)
    return X, y, act_type, pred_type

X_train, y_train, type_train, ptype_train = get_xyt(df_train)
X_val, y_val, type_val, ptype_val = get_xyt(df_val)
X_test, y_test, type_test, ptype_test = get_xyt(df_test)

# Save Bundle with Joblib
data_bundle = {
    'train': (X_train, y_train, type_train, ptype_train),
    'val': (X_val, y_val, type_val, ptype_val),
    'test': (X_test, y_test, type_test, ptype_test),
    'feature_names': feature_cols,
    'metadata': {
        'train_ids': df_train['question_id'].values,
        'val_ids': df_val['question_id'].values,
        'test_ids': df_test['question_id'].values
    }
}

# preview X_test, y_train, z_test
print("X_test:", X_test[:5])
print("y_test:", y_test[:5])
print("type_test:", type_test[:5])
print("ptype_test:", ptype_test[:5])

bundle_path = os.path.join(OUTPUT_PATH, "data_bundle.joblib")
joblib.dump(data_bundle, bundle_path)

print(f"Data bundle saved successfully to: {bundle_path}")
print(f"Train size: {len(X_train)}, Val size: {len(X_val)}, Test size: {len(X_test)}")

X_test: [[-0.38229254 -0.49062568 -0.4462975  -0.45905086 -0.47185686 -0.47978926
  -0.4742453  -0.54622114 -0.5839731  -0.5414086  -0.5434294  -0.5291126
  -0.5097144  -0.5680571  -0.47056147 -0.31664008 -0.4318952  -0.34217793
  -0.34955406 -0.21798216 -0.3612462  -0.34489903 -0.32870975 -0.31075075
  -0.25299394 -0.2906749  -0.35395163  0.19016466  0.35628816  0.34587023
   0.7410899  -0.3264896 ]
 [-0.56689173 -0.51018804 -0.5041776  -0.5392861  -0.5372618  -0.518143
  -0.5758942  -0.52279556 -0.5245444  -0.50608957 -0.5060818  -0.5006223
  -0.5428806  -0.4766334  -0.51951325 -0.4932412  -0.4878053  -0.47360644
  -0.42690784 -0.23622544 -0.16943404 -0.13196321  0.0288219   0.10888442
   0.16338544  0.1959943   0.23292832  0.30993918  0.48146853  0.5318404
   0.636531   -0.1714517 ]
 [-0.46767437 -0.45091686 -0.40649506 -0.40326822 -0.33999926 -0.50924826
  -0.37248418 -0.42982355 -0.32934225 -0.44874665 -0.34377104 -0.43192026
  -0.39157292 -0.32309905 -0.35340706 -0.42888695 -0.34